In [1]:
import torch
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split, Subset
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import CrossEntropyLoss

from sklearn.model_selection import train_test_split

import json
import random
import pickle
import csv
import numpy as np

import os
import datetime
import pandas as pd
from collections import Counter, defaultdict

In [2]:
#for good data sets, only thing to change
file_path_philipp = '/fast/AG_Metzger/philipp/python_files/gene_signature/mutual_information_approach/November_25_ReDo/preprocessing/pkl_files_filtered/'

data_set_name = 'treutlein_all_genes'


#input
hyper_params_input = data_set_name + '_test_accuracy_best_hyper_params.json'
model_weights_input = data_set_name + '_model_weights.pth'
test_data_set_input = data_set_name + '_test_dataset.pth'
barcode_NN_cont_mapping_input = data_set_name + '_barcode_NN_cont_mapping.pkl'
data_set_input = file_path_philipp + data_set_name + '.pkl'
#output
importance_scores_output = data_set_name + '_integrated_gradient.csv'



Overview of barcode_cont distribution

In [3]:
class CustomDataset(Dataset):
    # pca and occlusion manipulation outside of this class
    def __init__(self, dataframe, features = None, labels = None):
        self.dataframe = dataframe.copy()
        if features is not None and labels is not None:
            self.features = features
            self.labels = labels
        else:
            self.labels = self.dataframe['barcode_NN'].values
            self.features = self.dataframe.drop(['barcode_cont', 'barcode_NN'], axis=1).values

        
    def __len__(self):
        """
        Returns the total number of samples in the dataset.
        """
        return len(self.dataframe)
    
    def __getitem__(self, idx):
        sample = torch.tensor(self.features[idx], dtype=torch.float)  
        label = torch.tensor(self.labels[idx], dtype=torch.long)  
        return sample, label
    
    def num_classes(self):
        """
        Returns the number of unique classes in the dataset.
        """
        return np.unique(self.labels).size
        
    def num_features(self):
        """
        Returns the number of features for each sample in the dataset.
        """
        return self.features.shape[1]  

In [4]:
class Swish(nn.Module):
    def forward(self, x):
        return x * torch.sigmoid(x)

class CustomLayer(nn.Module):
    
    def __init__(self, size, dropout_rate, activation_func_name):
        super(CustomLayer, self).__init__()
        self.linear = nn.Linear(size, size)
        self.bn = nn.BatchNorm1d(size)
        self.dropout = nn.Dropout(dropout_rate)
        self.activation_func_name = activation_func_name  

        # Selecting the activation function
        if activation_func_name == 'relu':
            self.activation_func = nn.ReLU()
        elif activation_func_name == 'elu':
            self.activation_func = nn.ELU()
        elif activation_func_name == 'selu':
            self.activation_func = nn.SELU()
        elif activation_func_name == 'swish':
            self.activation_func = Swish()
        elif activation_func_name == 'mish':
            self.activation_func = nn.Mish()
        else:
            raise ValueError("Unsupported activation function")

        self.initialize_weights() 
            
    def initialize_weights(self):
        if self.activation_func_name in ['relu', 'swish', 'mish']:
            nn.init.kaiming_normal_(self.linear.weight, mode='fan_out', nonlinearity='relu')
        elif self.activation_func_name in ['elu', 'selu']:
            nn.init.kaiming_normal_(self.linear.weight, mode='fan_out', nonlinearity='leaky_relu')
        nn.init.constant_(self.linear.bias, 0)
         
    def forward(self, x):
        identity = x
        out = self.linear(x)

        out = self.bn(out)
        out = self.activation_func(out)
        out = self.dropout(out)
        out = out + identity  
        return out

class GeneExpressionModel(nn.Module):
         
    def __init__(self, input_dimension, num_classes, hidden_units, dropout_rate, num_layers, activation_function):
        super(GeneExpressionModel, self).__init__()

        # First layer to transform input dimension to hidden_units dimension
        self.initial_layer = nn.Linear(input_dimension, hidden_units)

        # Create a list of custom layers with residual connections
        self.layers = nn.ModuleList([CustomLayer(hidden_units, dropout_rate, activation_function) for _ in range(num_layers - 1)])
        
        # Output layer
        self.output = nn.Linear(hidden_units, num_classes) 

        with torch.no_grad():
            # Xavier initialization for the initial layer
            nn.init.kaiming_normal_(self.initial_layer.weight)
            nn.init.constant_(self.initial_layer.bias, 0)
           
            # Scaled Kaiming initialization for the output layer
            nn.init.kaiming_normal_(self.output.weight, mode='fan_out', nonlinearity='relu')
            self.output.weight.data *= 0.01  # Scaling factor
            nn.init.constant_(self.output.bias, 0)  

    def forward(self, x):
        # Pass the input through the initial layer
        x = self.initial_layer(x)

        # Pass the input through each custom layer
        for layer in self.layers:
            x = layer(x)

        # Output layer
        x = self.output(x)
        return x


Load the best neural network and the test data set that we will analyze.

In [5]:
with open(hyper_params_input, 'r') as f:
    best_hyperparameters = json.load(f)

# initialize model and data loaders
best_model = GeneExpressionModel(
    input_dimension=best_hyperparameters["data_overview"]["number_of_genes"],
    num_classes=best_hyperparameters["data_overview"]["number_of_barcodes"],
    hidden_units=best_hyperparameters["hyperparameters"]["hidden_units"],
    dropout_rate=best_hyperparameters["hyperparameters"]["dropout_rate"],
    num_layers=best_hyperparameters["hyperparameters"]["num_layers"],
    activation_function=best_hyperparameters["hyperparameters"]["activation_function"]
)

if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

test_dataset = torch.load(test_data_set_input)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)
best_model.load_state_dict(torch.load(model_weights_input))
best_model.to(device)


/tmp/ipykernel_3977465/1871324318.py:19: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  test_dataset = torch.load(test_data_set_input)
/tmp/ipykernel_3977465/1871324318.py:21

GeneExpressionModel(
  (initial_layer): Linear(in_features=20856, out_features=148, bias=True)
  (layers): ModuleList()
  (output): Linear(in_features=148, out_features=23, bias=True)
)

Here code for integrated gradients starts


In [6]:
def generate_scaled_inputs_and_gradients(model: torch.nn.Module, input: torch.Tensor, targets: torch.Tensor, steps: int = 200) -> torch.Tensor:
    model.eval()
    # we set the baseline to be zero but we do not need it actually for the computation.
    baseline = torch.zeros_like(input, device=device)  
    
    averaged_gradients = torch.zeros_like(input, device=device)
    # Interpolate scaled inputs and compute gradients
    for i in range(steps + 1):
        # Generate the scaled input
        scaled_input = (float(i) / steps) * input
        scaled_input.requires_grad_(True)
        
        output = model(scaled_input)
        loss = F.cross_entropy(output, targets)  
        model.zero_grad()
        loss.backward()
        input_gradients = scaled_input.grad

        assert input_gradients is not None
        averaged_gradients += input_gradients / steps

    scaled_averaged_integrated_gradients = averaged_gradients * input
        
    
    return scaled_averaged_integrated_gradients


In [7]:


# Initialize accumulators for gradient maps on the correct device
# Get the first input tensor to determine shape. and reinitialize test_loader
first_input_tensor = next(iter(test_loader))[0]  
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)
# 
gradient_maps_by_label = defaultdict(lambda: torch.zeros_like(first_input_tensor, device=device))

# Initialize a counter for the number of samples processed per label
num_samples_processed_by_label = defaultdict(int)
# batch size is 1, so we can iterate over the test_loader and accumulate the gradient maps directly
for input, target in test_loader:
    label = target.item()  

    # Move inputs and targets to the device
    input, target = input.to(device), target.to(device)

    # Generate gradient-activated maps
    gradient_map = generate_scaled_inputs_and_gradients(best_model, input, target, steps=200)

    gradient_maps_by_label[label] += gradient_map.squeeze()  # Accumulate directly

    # Track the number of samples per label
    num_samples_processed_by_label[label] += 1

# Calculate the mean gradient map for each label, ensuring final tensors are on CPU for further processing
mean_gradient_maps_by_label = {label: gradients.cpu() / num_samples_processed_by_label[label] for label, gradients in gradient_maps_by_label.items()}


/tmp/ipykernel_3977465/3412925630.py:21: DeprecationWarning: an integer is required (got type numpy.float64).  Implicit conversion to integers using __int__ is deprecated, and may be removed in a future version of Python.
  label = torch.tensor(self.labels[idx], dtype=torch.long)


In [9]:
gene_matrix = pd.read_pickle(data_set_input)
gene_names = list(gene_matrix.drop(['barcode_cont'], axis=1).columns)

with open(barcode_NN_cont_mapping_input, 'rb') as f:
    barcode_NN_cont_mapping = pickle.load(f)

with open(importance_scores_output, 'w', newline='') as csvfile:
    fieldnames = ['barcode_cont', 'gene', 'gene_index', 'score']
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)

    writer.writeheader()

    # ---------------------------------------------------------
    # 1. Write per-barcode signed scores (unchanged)
    # ---------------------------------------------------------
    for label, scores in mean_gradient_maps_by_label.items():
        gradients_np = scores.squeeze().cpu().numpy()
        original_barcode = barcode_NN_cont_mapping[label]

        for idx, (gene, score) in enumerate(zip(gene_names, gradients_np)):
            writer.writerow({
                'barcode_cont': original_barcode,
                'gene': gene,
                'gene_index': idx,
                'score': score
            })

    # ---------------------------------------------------------
    # 2. Compute global scores as MEAN ABSOLUTE IG across labels
    # ---------------------------------------------------------
    total_labels = len(mean_gradient_maps_by_label)
    global_abs_scores = np.zeros(len(gene_names), dtype=float)

    for _, scores in mean_gradient_maps_by_label.items():
        gradients_np = scores.squeeze().cpu().numpy()
        global_abs_scores += np.abs(gradients_np)

    global_abs_scores /= float(total_labels)

    # Sort genes by global importance
    sorted_indices = np.argsort(-global_abs_scores)

    # ---------------------------------------------------------
    # 3. Write global scores with barcode_cont = -1
    # ---------------------------------------------------------
    for idx in sorted_indices:
        writer.writerow({
            'barcode_cont': -1,
            'gene': gene_names[idx],
            'gene_index': idx,
            'score': global_abs_scores[idx]
        })
